<a href="https://colab.research.google.com/github/Ibrah-N/Deep-Learning-Projects-Computer-Vision/blob/main/dl_diffusers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# MODEL

In [1]:
import torch
import torch.nn as nn

import math

In [4]:
#==========================#
#  EMBEDDING - LAYER       #
#==========================#

class EmbeddingLayer(nn.Module):

  def __init__(self, vocab_size, d_model):
    super().__init__()

    self.vocab_size = vocab_size
    self.d_model = d_model

    self.embedding_layer = nn.Embedding(self.vocab_size, self.d_model)

  def forward(self, input):
    # (batch, seq_len) --> (batch, seq_len, d_model)
    return self.embedding_layer(input) * torch.sqrt(self.d_model)




#==================================#
#  POSITIONAL EMBEDDING - LAYER    #
#==================================#
class PositionalEmbeddingLayer(nn.Module):

  def __init__(self, d_model, seq_len, dropout):
    super().__init__()

    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(self.seq_len, self.d_model)

    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    # devision term
    div_term = torch.exp(torch.arange(0, self.d_model, 2) * (-torch.log(torch.tensor(10000.0)) / self.d_model))
    # sin to even indecies
    pe[:, 0::2] = torch.sin(position * div_term)
    # cos to odd indecies
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0)

    self.register_buffer('pe', pe)


  def forward(self, embedding):
    x = embedding + (self.pe[:, :embedding.shape[1], :]).requires_grad_(False)
    return self.dropout(x)



#==================================#
#  POSITIONAL EMBEDDING - LAYER    #
#==================================#
class PositionalEmbeddingLayer(nn.Module):

  def __init__(self, d_model, seq_len, dropout):
    super().__init__()

    self.d_model = d_model
    self.seq_len = seq_len
    self.dropout = nn.Dropout(dropout)

    pe = torch.zeros(self.seq_len, self.d_model)

    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    # devision term
    div_term = torch.exp(torch.arange(0, self.d_model, 2) * (-torch.log(torch.tensor(10000.0)) / self.d_model))
    # sin to even indecies
    pe[:, 0::2] = torch.sin(position * div_term)
    # cos to odd indecies
    pe[:, 1::2] = torch.cos(position * div_term)
    pe = pe.unsqueeze(0) # (1, seq_len, d_model)

    self.register_buffer('pe', pe)


  def forward(self, embedding):
    x = embedding + (self.pe[:, :embedding.shape[1], :]).requires_grad_(False)
    return self.dropout(x)



#==================================#
#    NORMALIZATION - LAYER      #
#==================================#
class NormalizationLayer(nn.Module):
  def __init__(self, features, eps=1e-6):
    super().__init__()

    self.features = features
    self.eps = eps

    self.gamma = nn.Parameter(torch.ones(features))
    self.beta = nn.Parameter(torch.zeros(features))

  def forward(self, x):
    # x: seq_len, d_model
    mean = x.mean(-1, keepdim=True)
    std = x.std(-1, keepdim=True)

    return self.gamma * (x - mean) / (std + self.eps) + self.beta



#==================================#
#    FeedForward - LAYER      #
#==================================#
class FeedForwardLayer(nn.Module):
  def __init__(self, d_model, d_ff, dropout: float):
    super().__init__()

    self.linear_1 = nn.Linear(d_model, d_ff)
    self.dropout = nn.Dropout(dropout)
    self.linear_2 = nn.Linear(d_ff, d_model)

  def forward(self, x):
    x = self.linear_1(x)
    x = self.dropout(torch.relu(x))
    x = self.linear_2(x)
    return x


#==================================#
#    MultiheadAttention - LAYER    #
#==================================#
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads, dropout: float):
    super().__init__()

    self.d_model = d_model
    self.n_heads = n_heads

    assert d_model % n_heads == 0, "d_model must be divisible by h"

    self.d_k = self.d_model // self.n_heads

    self.w_q = nn.Linear(d_model, d_model, bias=False)
    self.w_k = nn.Linear(d_model, d_model, bias=False)
    self.w_v = nn.Linear(d_model, d_model, bias=False)
    self.w_o = nn.Linear(d_model, d_model, bias=False)

    self.dropout = nn.Dropout(dropout)


  @staticmethod
  def attention(query, key, value, mask, dropout: nn.Dropout):
    d_k = query.shape[-1]

    # batch, h, seq_len, d_k --> batch, h, seq_len, seq_len
    scores = (query @ key.transpose(-2, -1)) / math.sqrt(d_k)

    if mask is not None:
      scores = scores.masked_fill_(mask == 0, -1e9)

    scores = scores.softmax(dim=-1)

    if dropout is not None:
      scores = dropout(scores)

    # batch, h, seq_len, seq_len --> batch, h, seq_len, d_k
    # scores: batch, h, seq_len, seq_len, value: batch, h, seq_len, d_k
    # scores @ value --> batch, h, seq_len, d_k
    return scores @ value, scores



  def forward(self, q, k, v, mask=None):
    query = self.w_q(q) # q: batch, seq_len, d_model -->  batch, seq_len, d_model
    key = self.w_v(k) # k: batch, seq_len, d_model -->  batch, seq_len, d_model
    value = self.w_k(v)   # v: batch, seq_len, d_model -->  batch, seq_len, d_model

    # (batch, seq_len, d_model) --> (batch, seq_len, h, d_k) --> transpose --> (batch, h, seq_len, d_k)
    query = query.view(query.shape[0], query.shape[1], self.n_heads, self.d_k).transpose(1, 2)
    key = key.view(key.shape[0], key.shape[1], self.n_heads, self.d_k).transpose(1, 2)
    value = value.view(value.shape[0], value.shape[1], self.n_heads, self.d_k).transpose(1, 2)

    # calculate attention scores
    x, self.attention_scores = MultiHeadAttention.attention(query, key, value, mask, self.dropout)

    # x: batch, h, seq_len, d_k --> batch, seq_len, h, d_k --> batch, seq_len, d_model
    x = x.transpose(1, 2).contiguous().view(x.shape[0], -1, self.n_heads * self.d_k)

    # Multiply by Wo
    # (batch, seq_len, d_model) --> (batch, seq_len, d_model)
    return self.w_o(x)



#==================================#
#    ResidualConnection - LAYER    #
#==================================#
class ResidualConnection(nn.Module):
  def __init__(self, features: int, dropout: float):
    super().__init__()

    self.dropout = nn.Dropout(dropout)
    self.norm  = NormalizationLayer()


  def forward(self, x, sublayer):
    return x + self.dropout(sublayer(self.norm(x)))


#==================================#
#    EncoderBlock   -  LAYER       #
#==================================#
class EncoderBlock(nn.Module):
  def __init__(self, features: int,
               attention_layer: MultiHeadAttention,
               feedforward_layer: FeedForwardLayer,
               dropout: int
               ):
    super().__init__()

    self.attention_layer = attention_layer
    self.feedforward_layer = feedforward_layer
    self.residual_connection = nn.ModuleList(
                                ResidualConnection(features, dropout)
                                for _ in range(2))


  def forward(self, x, src_mask):
    x = self.residual_connection[0](x, lambda x:self.attention_layer(x, x, x, src_mask))
    x = self.residual_connection[1](x, lambda x:self.feedforward_layer)

    return x



#==================================#
#    Encoder  -  LAYER             #
#==================================#
class Encoder(nn.Module):
  def __init__(self, features: int, layers: nn.ModuleList):
    super().__init__()
    self.layers = layers
    self.norm = NormalizationLayer(features)

  def forward(self, x, mask):
    for layer in self.layers:
      x = layer(x, mask)

    return self.norm(x)




#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++#
#+++++++++++++++        D E C O D E R        +++++++++++++++#
#+++++++++++++++++++++++++++++++++++++++++++++++++++++++++++#

#==================================#
#    DecoderLayer  -  LAYER        #
#==================================#
class DecoderBlock(nn.Module):
  def __init__(self, features: int,
               self_attention: MultiHeadAttention,
               cross_attention: MultiHeadAttention,
               feedfowrad_layer: FeedForwardLayer,
               dropout: float):
    super().__init__()

    self.self_attention = self_attention
    self.cross_attention = cross_attention
    self.feedforward_layer = feedfowrad_layer

    self.norm = NormalizationLayer(features)
    self.residual_connections = nn.ModuleList(
                                ResidualConnection(features, dropout)
                                for _ in range(3))

  def forward(self, x, encoder_output, src_mask, tgt_mask):
    x = self.residual_connections[0](x, lambda x: self.attention(x, x, x, tgt_mask))
    x = self.residual_connections[1](x, lambda x: self.attention(x, encoder_output,
                                                                 encoder_output, src_mask))
    x = self.residual_connections[2](x, self.feedforward_layer)

    return self.norm(x)


#==================================#
#    Decoder  -  LAYER             #
#==================================#
class Decoder(nn.ModuleList):
  def __init__(self, features: int, layers: nn.ModuleList):
    super().__init__()

    self.layers = layers
    self.norm = NormalizationLayer(features)


  def forward(self, x, encoder_output, src_mask, tgt_mask):
    for layer in self.layers:
      x = layer(x, encoder_output, src_mask, tgt_mask)

    return self.norm(x)


#==================================#
#    Projection  -  LAYER          #
#==================================#
class ProjectLayer(nn.Module):
  def __init__(self, d_model, vocab_size):
    super().__init__()

    self.linear = nn.Linear(d_model, vocab_size)


  def forward(self, x):
    # (batch, seq_len, d_model) --> (batch, seq_len, vocab_size)
    return self.linear(x)



class Transformer(nn.Module):
  def __init__(self, encoder: Encoder,
               decoder: Decoder,
               src_embed: EmbeddingLayer,
               tgt_embed: EmbeddingLayer,
               src_pos_embed: PositionalEmbeddingLayer,
               tgt_pos_embed: PositionalEmbeddingLayer,
               projector: ProjectLayer
               ):
    super().__init__()

    self.encoder = encoder
    self.decoder = decoder
    self.src_embed = src_embed
    self.tgt_embed = tgt_embed
    self.src_pos_embed = src_pos_embed
    self.tgt_pos_embed = tgt_pos_embed
    self.projector = projector



  def encode(self, x, src_mask):
    # (batch, seq_len, d_model)
    x = self.src_embed(x)
    x = self.src_pos_embed(x)
    return self.encoder(x, src_mask)


  def decode(self,
             encoder_output: torch.Tensor, src_mask: torch.Tensor,
             target_input: torch.Tensor, tgt_mask: torch.Tensor):
    # (batch, seq_len, d_model)
    x = self.tgt_embed(x)
    x = self.tgt_pos_embed(x)
    return self.decoder(x)


  def project(self, x):
    # batch, seq_len, vocab_size)
    return self.projector(x)



def build_transformer(
    self, src_vocab_size: int, tgt_vocab_size: int,
    src_seq_len: int, tgt_seq_len: int,
    d_model: int = 512, n_heads: int = 4, num_encoder_layers: int = 6,
    num_decoder_layers: int = 6, d_ff: int = 2048, dropout: float = 0.1):


    # initialize embedding layers
    src_embed = EmbeddingLayer(src_vocab_size, d_model)
    tgt_embed = EmbeddingLayer(tgt_vocab_size, d_model)

    # positional embedding layers
    src_pos_embed = PositionalEmbeddingLayer(d_model, src_seq_len, dropout)
    tgt_pos_embed = PositionalEmbeddingLayer(d_model, tgt_seq_len, dropout)


    # initialize encoder
    encoder_layers = []
    for _ in range(num_encoder_layers):
      encoder_layer = MultiHeadAttention(d_model=d_model, n_heads=n_heads, dropout = dropout)
      feedforward_layer = FeedForwardLayer(d_model=d_model, d_ff=d_ff, dropout=dropout)
      encoder_block = EncoderBlock(features=d_model, attention_layer=encoder_layer,
                                   feedforward_layer=feedforward_layer, dropout=dropout)

      encoder_layers.append(encoder_block)


    # initialize decoder
    for _ in range(num_decoder_layers):
      decoder_layer = MultiHeadAttention(d_model=d_model, n_heads=n_heads, dropout = dropout)
      cross_attention = MultiHeadAttention(d_model=d_model, n_heads=n_heads, dropout=dropout)
      feedforward_layer = FeedForwardLayer(d_model=d_model, d_ff=d_ff, dropout=dropout)

    # sub models (encoder and decoder)
    encoder_model = Encoder(features=d_model, layers=nn.ModuleList(encoder_layers))
    decoder_model = Decoder(features=d_model, layers=nn.ModuleList(decoder_layers))

    # initialize projecter
    projector = ProjectLayer(d_model=d_model, vocab_size=tgt_vocab_size)


    # transformer
    transformer = Transformer(
               encoder=encoder_model,
               decoder=decoder_model,
               src_embed=src_embed,
               tgt_embed=tgt_embed,
               src_pos_embed=src_pos_embed,
               tgt_pos_embed=tgt_pos_embed,
               projector=projector
               )

    for p in transformer.parameters():
      if p.dim() > 1:
        nn.init.xavier_uniform_(p)

    return transformer